# 04 · 평가 (held-out) — 도메인 QA / instruction

**TL;DR** — 파인튜닝한 endpoint를 held-out 세트로 평가해 성공 기준을 수치로 확인합니다.

**Why** — 학습이 실제로 효과가 있었는지는 학습에 쓰지 않은 held-out 지표로만 판단할 수 있습니다. 합성 데이터로 평가하면 teacher 모델을 얼마나 모방했는지를 재는 데 그칩니다.

**기존 Pain Point** — 합성 데이터나 학습셋으로 평가하면 성능이 과대평가됩니다. 반드시 시드의 test 스플릿, 또는 증강 이전에 분리해 둔 슬라이스만 사용해야 합니다.

> 🔴 실제 실행 시 AWS 자격증명·GPU·엔드포인트 과금이 발생합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
# 평가 메트릭 의존성 (pyproject 코어에도 있음; 미설치 환경 대비 안전망). uv 우선, pip 폴백
import shutil, subprocess, sys
_pkgs = ['scikit-learn>=1.5.0', 'rouge-score>=0.1.2', 'rapidfuzz>=3.9.0']
if shutil.which('uv'):
    subprocess.run(['uv', 'pip', 'install', '--python', sys.executable, '-q', *_pkgs], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_pkgs], check=True)
print('installed:', _pkgs)

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

In [ ]:
import os, importlib
from common import config, aws_utils, gemma_format as gf; importlib.reload(config)
import importlib, track_data as td; importlib.reload(td)
# 트랙 전용 키 우선 — 전역 endpoint_name 은 다른 트랙이 덮어씁니다.
%store -r ep_domain_qa
%store -r endpoint_name
endpoint_name = globals().get('ep_domain_qa') or globals().get('endpoint_name')
assert endpoint_name, 'endpoint_name 이 없습니다 — 03의 배포 셀을 먼저 실행하세요.'
print('사용할 endpoint:', endpoint_name)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(config.DEFAULT_MODEL_ID, token=config.get_hf_token())
# held-out 개수: 튜토리얼 기본 50(단건 서빙 기준 ~1~2분). 정식 벤치는 N_EVAL을 키우세요(env로도 조정 가능).
N_EVAL = 20 if config.is_dry_run() else int(os.environ.get('N_EVAL', '50'))
ENGINE = config.SERVING_ENGINE   # 로그 표시용(vllm/sglang/lmi 모두 messages 스키마라 호출 코드는 동일)

## 1. held-out 세트 로드 (🔴 합성/학습셋 아님)
평가에는 **학습에 쓰지 않은** 예시만 사용합니다. `01_data_and_synthetic`은 시드의 **앞** `config.NUM_SEED_SAMPLES`건(기본 300)을 학습 JSONL에 넣으므로, 여기서는 그 **뒤** 인덱스에서 `N_EVAL`건을 잘라 씁니다.

🔴 **넉넉히 로드해서 뒤쪽 N건을 쓰는 방식(`pool[-N_EVAL:]`)은 위험합니다.** 예를 들어 `N_EVAL=50`이면 50×3=150건만 로드되어 held-out이 학습 구간(0~299) **안쪽**에 통째로 들어가고, 결과적으로 학습 데이터로 평가해 점수가 부풀려집니다. 그래서 아래 셀은 학습 구간 건수를 명시적으로 건너뜁니다.

`load_seed_examples`는 같은 인덱스를 항상 같은 순서로 돌려주므로(분류 트랙은 고정 시드 42로 셔플) 이 분리는 재현 가능합니다.

In [ ]:
# 🔴 학습 구간과 겹치지 않게 분리: 01이 학습에 쓴 '앞 N_TRAIN_USED건'을 건너뛰고 그 뒤 N_EVAL건을 쓴다.
N_TRAIN_USED = config.NUM_SEED_SAMPLES   # 01_data_and_synthetic 이 학습에 사용한 앞부분 건수
pool = td.load_seed_examples(N_TRAIN_USED + N_EVAL, token=config.get_hf_token())
heldout = pool[N_TRAIN_USED:N_TRAIN_USED + N_EVAL]
assert heldout, (
    f'시드가 {len(pool)}건뿐이라 학습 구간({N_TRAIN_USED}건) 뒤에 남는 예시가 없습니다 — '
    'NUM_SEED_SAMPLES를 줄이거나 더 큰 시드 데이터셋을 쓰세요.')
print(f'held-out: {len(heldout)}건  (시드 인덱스 {N_TRAIN_USED}~{N_TRAIN_USED + len(heldout) - 1}'
      f' — 학습 구간 0~{N_TRAIN_USED - 1} 제외)')

## 2. endpoint로 예측 생성 (🔴 sagemaker-runtime, 소규모 병렬)
held-out 각 입력을 endpoint로 호출해 예측을 모읍니다. 재현성을 위해 `temperature=0.0`(결정론적)으로 디코딩합니다.
- **호출 스키마**: vLLM/SGLang/LMI 모두 `{messages}`를 받고 **서버가 chat template을 적용**하므로 우리가 렌더할 필요가 없습니다(엔진을 바꿔도 이 셀은 그대로 돕니다).
- **소규모 병렬**(`ThreadPoolExecutor`, 순서 보존)로 왕복 지연을 겹칩니다. 세 엔진 모두 연속 배칭이 있어 동시성 8로 둡니다 — endpoint 인스턴스가 작아 429/타임아웃이 나면 낮추세요.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def _predict(ex):
    # 🔴 vllm/sglang/lmi 공통: messages 그대로 전송 → 서버가 chat template 적용.
    msgs = gf.build_inference_messages(ex['input'], system_content=td.SYSTEM_PROMPT)
    return aws_utils.invoke_sagemaker_chat(
        endpoint_name, msgs, region=config.AWS_REGION, max_tokens=512, temperature=0.0)

MAX_WORKERS = int(os.environ.get('EVAL_WORKERS', '8'))   # 연속 배칭 엔진이라 8부터 시작
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    preds = list(pool.map(_predict, heldout))   # map은 입력 순서 보존 → heldout[i]와 preds[i] 매칭 유지
print('predictions generated:', len(preds), f'(engine={ENGINE}, workers={MAX_WORKERS})')

## 3. 메트릭 (도메인QA): **Bedrock LLM-judge**(primary, correctness/helpfulness/groundedness 1-5) + ROUGE-L proxy
도메인 QA는 정답이 자유형 문장이라 exact-match로는 제대로 평가할 수 없으므로, correctness·helpfulness·groundedness를 1~5점으로 채점하는 **Bedrock LLM-judge**를 주 지표로 사용하고, ROUGE-L은 보조 proxy로만 함께 봅니다. dolly 데이터셋은 train 스플릿만 제공하므로, 결정론적으로 분리한 슬라이스를 held-out으로 사용합니다.

In [ ]:
from common import eval_utils, config
rouge = eval_utils.eval_rouge([(pred, ex['output']) for pred, ex in zip(preds, heldout)])
print('ROUGE-L proxy:', rouge)
judged = []
for pred, ex in list(zip(preds, heldout))[:20]:
    judged.append(eval_utils.llm_judge(
        model_id=config.BEDROCK_CLAUDE_MODEL_ID, region=config.BEDROCK_REGION,
        source=ex['input'], prediction=pred, reference=ex['output'],
        rubric='Rate the answer for correctness, helpfulness, and (if context present) groundedness.',
        axes=['correctness', 'helpfulness', 'groundedness']))
print('LLM-judge:', eval_utils.aggregate_judge(judged, ['correctness','helpfulness','groundedness']))

✅ 평가가 끝났습니다. 이 지표를 파인튜닝 전 baseline gemma와 비교하면 학습으로 얻은 개선폭을 정량적으로 확인할 수 있습니다. 다음은 **05_agentic_strands.ipynb**로 endpoint를 tool 삼아 agentic 루프를 구성합니다.

> 💡 참고: SageMaker SDK v3의 관리형 evaluator(`BenchMarkEvaluator`/`LLMAsJudgeEvaluator`/`CustomScorerEvaluator`)는 **SageMaker Public Hub에 평가 레시피가 등록된 모델(Amazon Nova·일부 JumpStart)** 전용입니다. gemma-4 커스텀 파인튜닝 산출물(S3 체크포인트)은 Hub 레시피가 없어 지원되지 않으므로(실측: `DescribeHubContent ... does not exist`), 이 킷은 위의 **로컬 메트릭 평가**를 gemma-4의 평가 경로로 사용합니다.